# Riparian hotspots — statewide run, on Colab

Cell 1 installs, cell 2 points at your code, cells 3–6 run.

**Put the five `fl_*.py` files somewhere on Drive** (a `code/` folder beside
your project folder is the usual arrangement) and set `CODE_DIR` in cell 2.

Nothing in this notebook makes a decision. Every switch is in `fl_config.py`,
and `COLAB` in §0 is the one that puts the build on local disk and the finished
copy back on Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# 1. Dependencies. Colab has most of these; pyogrio is the one that matters,
#    because it is what pushes the state filter down into the national CSB file.
!pip -q install geopandas pyogrio mapclassify

# 2. Set script and project directories.

import os, sys

CODE_DIR    = '/content/drive/MyDrive/FergusonLynch/03_AtRiskSpeciesData/G3S3_update_EF/Parcel_priority_maps_KY/scripts'          # the fl_*.py files
PROJECT_DIR = '/content/drive/MyDrive/FergusonLynch/03_AtRiskSpeciesData/G3S3_update_EF/Parcel_priority_maps_KY'

os.environ['FL_PROJECT_DIR'] = PROJECT_DIR
os.environ['FL_STATE_FIPS']  = '21'        # '47' Tennessee, '01' Alabama, ...
# os.environ['FL_COUNTIES']  = '047,225'   # uncomment for a two-county test run

sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2b. Stage the big read-only files onto local disk.
#
# Reading a GeoPackage is SQLite doing thousands of small random reads, and
# Drive is a FUSE mount where each one is a network round trip. One sequential
# copy is the access pattern Drive IS good at; everything after it is local SSD.
#
# MUST run before `from fl_config import ...` — fl_config resolves CACHE at
# import time and prefers /content/clipped_layers when the file is there.
# Idempotent: copies only what is missing, deletes nothing.

import glob, shutil, time

SRC_CLIP = os.path.join(PROJECT_DIR, 'clipped_layers')
DST_CLIP = '/content/clipped_layers'
SRC_SURF = os.path.join(PROJECT_DIR, 'outputs_statewide', '_state', 'output')
DST_SURF = '/content/outputs_statewide/_state/output'


def stage(src, dst):
    name = os.path.basename(dst)
    if not os.path.exists(src):
        print(f'  not on Drive     {name}'); return
    if os.path.exists(dst):
        print(f'  already local    {name}'); return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    mb, t0 = os.path.getsize(src) / 1e6, time.time()
    shutil.copy2(src, dst)
    ok = os.path.getsize(dst) == os.path.getsize(src)
    print(f'  copied {mb:9,.0f} MB in {time.time() - t0:5.0f}s  {name}'
          + ('' if ok else '   <- SIZE MISMATCH: delete it and re-run'))


print('staging read-only inputs to local disk')
for _p in sorted(glob.glob(os.path.join(SRC_CLIP, '*_state_layers.gpkg'))):
    stage(_p, os.path.join(DST_CLIP, os.path.basename(_p)))
for _p in sorted(glob.glob(os.path.join(SRC_SURF, 'state_surface.*'))):
    stage(_p, os.path.join(DST_SURF, os.path.basename(_p)))

# Now, and only now, import.
from fl_config import describe, CACHE
describe()
assert CACHE.startswith('/content/clipped_layers'), (
    f'cache is still being read over Drive:\n  {CACHE}\n'
    f'the copy above did not land where fl_config looks')

staging read-only inputs to local disk
  already local    ky_state_layers.gpkg
  already local    state_surface.gpkg
  already local    state_surface.json
CONFIGURATION
  running on   Google Colab  (Drive mounted, building on local disk)
  project      /content/drive/MyDrive/FergusonLynch/03_AtRiskSpeciesData/G3S3_update_EF/Parcel_priority_maps_KY
  raw inputs   /content/drive/MyDrive/FergusonLynch/03_AtRiskSpeciesData/G3S3_update_EF/Parcel_priority_maps_KY/raw_data
  cache        /content/clipped_layers/ky_state_layers.gpkg
  outputs      /content/outputs_statewide
               -> copied at the end to /content/drive/MyDrive/FergusonLynch/03_AtRiskSpeciesData/G3S3_update_EF/Parcel_priority_maps_KY/outputs_statewide
  switches     RUN_PREPROCESS='auto'  RUN_SURFACE='auto'  RUN_PASS_C=True  SURFACE_GUARD=True
  study area   state 21 (Kentucky, KY) · counties all · extent whole_county
  surface      1-mile cells · 3-mile band · 100 ft buffer · 40 km tiles
  threshold    basis county · r

## Preprocessing — run once

Cuts the national and statewide downloads to the state and writes one cached
GeoPackage. Tens of minutes, almost all of it reading CSB. Once the cache
exists, `RUN_PREPROCESS = 'auto'` skips everything that is still current.

In [3]:
%run fl_01_preprocess.py

## The statewide analysis

Pass A (the density surface) is the expensive one and checkpoints every tile,
so a runtime reset costs one tile rather than the pass. Everything after Pass A re-runs in minutes
off the cached surface.

If the runtime drops, re-run cells 1–2b and then this cell; it picks up where it
stopped.

In [ ]:
%run fl_02_statewide.py

Output hidden; open in https://colab.research.google.com to view.

## Output location

`outputs_statewide/` in your project folder on Drive:

* `county_manifest.csv` — the run ledger. Start here.
* `_state/` — the surface, the state figures, the validation tables.
* `counties/<FIPS>_<Name>/` — one folder per county.

The build happens on `/content` (local disk) and is copied to Drive at the end,
because a GeoPackage written straight to a FUSE mount can lose layers without
raising.